In [8]:
# =========================================
# 📦 IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import joblib

# =========================================
# 📂 LOAD DATA
# =========================================
df = pd.read_csv("QCFE5_Perfect_Customer_Segmentation.csv")

# =========================================
# 🧹 CLEAN COLUMN NAMES
# =========================================
df.columns = df.columns.str.strip()

# =========================================
# 📅 DATE PROCESSING
# =========================================
df['Order_date'] = pd.to_datetime(df['Order_date'], errors='coerce')

# =========================================
# 🧠 CREATE RFM FEATURES (MOST IMPORTANT)
# =========================================

# Recency (days since last order)
df['Recency_New'] = (df['Order_date'].max() - df['Order_date']).dt.days

# Frequency (use Total_Orders directly if available)
df['Frequency_New'] = df['Total_Orders']

# Monetary (total spend)
df['Monetary_New'] = df['Total_Spend']

# =========================================
# 🎯 FINAL FEATURE SET (OPTIMIZED)
# =========================================
features = [
    'Recency_New',
    'Frequency_New',
    'Monetary_New',
    'Items_Count',
    'Customer_Rating',
    'Discount_Usage_Rate'
]

# =========================================
# 🛑 CHECK MISSING COLUMNS
# =========================================
missing = [col for col in features if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# =========================================
# 📊 PREPARE DATA
# =========================================
X = df[features].copy()
X.fillna(X.mean(), inplace=True)

# =========================================
# 🔥 LOG TRANSFORMATION (IMPORTANT)
# =========================================
for col in ['Monetary_New', 'Frequency_New', 'Recency_New']:
    X[col] = np.log1p(X[col])

# =========================================
# 🔢 SCALING
# =========================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================================
# ⚡ SAMPLE FOR FAST TUNING
# =========================================
sample_size = 20000
idx = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[idx]

# =========================================
# 🎯 FIND BEST K
# =========================================
best_k = 2
best_score = -1

print("\n🔍 Finding Best K...")

for k in range(2, 8):
    km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=10000)
    labels = km.fit_predict(X_sample)

    score = silhouette_score(X_sample, labels)
    print(f"K={k}, Silhouette={score:.4f}")

    if score > best_score:
        best_score = score
        best_k = k

print(f"\n✅ Best K selected: {best_k}")

# =========================================
# 🤖 FINAL MODEL (KMEANS)
# =========================================
kmeans = MiniBatchKMeans(
    n_clusters=best_k,
    random_state=42,
    batch_size=10000
)

labels_kmeans = kmeans.fit_predict(X_scaled)

# =========================================
# 🤖 OPTIONAL: GMM MODEL (BETTER FOR OVERLAP)
# =========================================
gmm = GaussianMixture(n_components=best_k, random_state=42)
labels_gmm = gmm.fit_predict(X_scaled)

# Choose best (usually GMM if overlap exists)
labels = labels_gmm

df['Customer_Segment'] = labels

# =========================================
# 📊 FINAL EVALUATION
# =========================================
idx = np.random.choice(len(X_scaled), 30000, replace=False)

X_eval = X_scaled[idx]
labels_eval = labels[idx]

sil_score = silhouette_score(X_eval, labels_eval)

print("\n📊 Final Model Score:")
print(f"Silhouette Score: {sil_score:.4f}")

# =========================================
# 🏷️ SEGMENT LABELING (DYNAMIC)
# =========================================
segment_summary = df.groupby('Customer_Segment')[features].mean()

print("\n📊 Segment Summary:")
print(segment_summary)

# =========================================
# 💾 SAVE MODEL
# =========================================
joblib.dump(kmeans, "kmeans_model.pkl")
joblib.dump(gmm, "gmm_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")

df.to_csv("segmented_customers.csv", index=False)

print("\n✅ Improved Model Trained & Saved Successfully!")


🔍 Finding Best K...
K=2, Silhouette=0.2184
K=3, Silhouette=0.1855
K=4, Silhouette=0.2038
K=5, Silhouette=0.1820
K=6, Silhouette=0.1661
K=7, Silhouette=0.1537

✅ Best K selected: 2

📊 Final Model Score:
Silhouette Score: 0.1907

📊 Segment Summary:
                  Recency_New  Frequency_New  Monetary_New  Items_Count  \
Customer_Segment                                                          
0                   49.809376       6.740076   8338.471047     9.966942   
1                   49.946161      10.642918  13267.107535    10.017640   

                  Customer_Rating  Discount_Usage_Rate  
Customer_Segment                                        
0                        3.884428             0.354290  
1                        3.878632             0.346487  

✅ Improved Model Trained & Saved Successfully!
